<a href="https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/tutorials/quickstart/build_RAG_with_milvus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>   <a href="https://github.com/milvus-io/bootcamp/blob/master/tutorials/quickstart/build_RAG_with_milvus.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on%20GitHub-555555?style=flat&logo=github&logoColor=white" alt="GitHub Repository"/>

# Build RAG with Milvus

<img src="https://raw.githubusercontent.com/milvus-io/bootcamp/master/tutorials/quickstart/apps/rag_search_with_milvus/pics/rag_demo.png"/>

In this tutorial, we will show you how to build a RAG(Retrieval-Augmented Generation) pipeline with Milvus.

The RAG system combines a retrieval system with a generative model to generate new text based on a given prompt. The system first retrieves relevant documents from a corpus using Milvus, and then uses a generative model to generate new text based on the retrieved documents.


## Preparation
### Dependencies and Environment

In [1]:
! pip install PyPDF2 sentence-transformers pymilvus langchain tqdm PyMuPDF tools pdfplumber

> If you are using Google Colab, to enable dependencies just installed, you may need to **restart the runtime** (click on the "Runtime" menu at the top of the screen, and select "Restart session" from the dropdown menu).

We will use OpenAI as the LLM in this example. You should prepare the [api key](https://platform.openai.com/docs/quickstart) `OPENAI_API_KEY` as an environment variable.

### Prepare the data

We use the FAQ pages from the [Milvus Documentation 2.4.x](https://github.com/milvus-io/milvus-docs/releases/download/v2.4.6-preview/milvus_docs_2.4.x_en.zip) as the private knowledge in our RAG, which is a good data source for a simple RAG pipeline.

Download the zip file and extract documents to the folder `milvus_docs`.

We load all markdown files from the folder `milvus_docs/en/faq`. For each document, we just simply use "# " to separate the content in the file, which can roughly separate the content of each main part of the markdown file.

### Prepare the Embedding Model

We initialize the OpenAI client to prepare the embedding model.

Define a function to generate text embeddings using OpenAI client. We use the [text-embedding-3-small](https://platform.openai.com/docs/guides/embeddings) model as an example.

In [2]:
from pymilvus import MilvusClient, DataType
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import PyPDF2

In [ ]:
# Replace with your Zilliz Cloud details
URI = "https://in03-56a0ce.serverless.aws-eu-central-1.cloud.zilliz.com"
TOKEN = "f0cef636066c8653aa07cf9b435a2faf18dcb7a1e13a19a8c61c11b786f8a49d61c6a7d9d883784"  # e.g., "root:your-secret-password"

# Initialize MilvusClient with a local database file
def create_milvus_client(uri, token):
    client = MilvusClient(
        uri=uri,
        token=token,
        secure=True
    )
    print("Connected to Milvus Online!")
    return client
client = create_milvus_client(URI, TOKEN)

Connected to Milvus Online!


In [4]:
collections = client.list_collections()
collections

['rag_vulnbot_milvus']

In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)

In [10]:
def create_collection(collection_name):
  if not client.has_collection(collection_name):
    schema = client.create_schema(
      auto_id=True,          # Milvus generates IDs
      enable_dynamic_field=False  # We want fixed schema
    )

    # Add ALL fields to schema
    schema.add_field("id", DataType.INT64, is_primary=True)
    schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=384)
    schema.add_field("text", DataType.VARCHAR, max_length=65535)
    schema.add_field("book_title", DataType.VARCHAR, max_length=512)
    schema.add_field("chunk_order", DataType.INT32)
    schema.add_field("section_id", DataType.VARCHAR, max_length=256)

    # ✅ CORRECT: Pass schema to create_collection
    client.create_collection(
      collection_name=collection_name,
      schema=schema,
      consistency_level="Strong"
    )
    if(not client.has_collection(collection_name)):
      print("Create collection failed")
      return None

    # Create index
    index_params = client.prepare_index_params(
      field_name="embedding",
      index_type="AUTOINDEX",
      metric_type="COSINE"
    )
    client.create_index(collection_name, index_params)
    client.load_collection(collection_name)

    print(f"✅ Collection '{collection_name}' created with metadata fields.")
    return 1

Generate a test embedding and print its dimension and first few elements.

## Load data into Milvus

### Create the Collection

> As for the argument of `MilvusClient`:
> - Setting the `uri` as a local file, e.g.`./milvus.db`, is the most convenient method, as it automatically utilizes [Milvus Lite](https://milvus.io/docs/milvus_lite.md) to store all data in this file.
> - If you have large scale of data, you can set up a more performant Milvus server on [docker or kubernetes](https://milvus.io/docs/quickstart.md). In this setup, please use the server uri, e.g.`http://localhost:19530`, as your `uri`.
> - If you want to use [Zilliz Cloud](https://zilliz.com/cloud), the fully managed cloud service for Milvus, adjust the `uri` and `token`, which correspond to the [Public Endpoint and Api key](https://docs.zilliz.com/docs/on-zilliz-cloud-console#free-cluster-details) in Zilliz Cloud.

Check if the collection already exists and drop it if it does.

Create a new collection with specified parameters.

If we don't specify any field information, Milvus will automatically create a default `id` field for primary key, and a `vector` field to store the vector data. A reserved JSON field is used to store non-schema-defined fields and their values.

### Insert data
Iterate through the text lines, create embeddings, and then insert the data into Milvus.

Here is a new field `text`, which is a non-defined field in the collection schema. It will be automatically added to the reserved JSON dynamic field, which can be treated as a normal field at a high level.

## Build RAG

### Retrieve data for a query

Let's specify a frequent question about Milvus.

In [28]:
# Search with context
def search_with_context(query, context_window=2, limit=1000):
    client = create_milvus_client(URI, TOKEN)
    collections = client.list_collections()
    query_emb = model.encode([query], convert_to_tensor=False).tolist()

    retrieved_info = ""
    for collection_name in collections:
      # 1. Find relevant chunk
      results = client.search(
          collection_name=collection_name,
          data=query_emb,
          limit=1,
          output_fields=["chunk_order", "section_id", "book_title"],
          search_params={"metric_type": "COSINE", "params": {"nprobe": 10}}
      )

      if not results or not results[0]:  # Check if results or results[0] is empty
          print(f"No results found in collection: {collection_name}")
          continue # Skip to the next collection if no results

      hit = results[0][0]
      order = hit["entity"]["chunk_order"]
      section = hit["entity"]["section_id"]

      # 2. Get neighbors using query()
      neighbor_orders = list(range(order - context_window, order + context_window + 1))
      neighbor_orders = [o for o in neighbor_orders if o >= 0]

      neighbors = client.query(
          collection_name=collection_name,
          filter=f"section_id == '{section}' and chunk_order in {neighbor_orders}",
          output_fields=["text", "chunk_order"],
          offset=0,
          limit=10
      )

      # Sort by order
      neighbors.sort(key=lambda x: x["chunk_order"])
      retrieved_info += "\n".join([n["text"] for n in neighbors]) + f"\nSource: {collection_name}"
    return retrieved_info[:limit]

In [1]:
!pip install tools pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 110.0 MB/s eta 0:00:00


In [15]:
import pdfplumber

def extract_text_from_pdf(pdf_path, book_title):
    """Reliable PDF text extraction - NO fitz DEPENDENCIES"""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    except Exception as e:
        if "Password required" in str(e):
            raise ValueError("PDF is encrypted! Try: pdfplumber.open(path, password='your_pass')") from e
        raise RuntimeError(f"PDF processing failed: {str(e)}") from e

# TEST FIRST - NO FITZ INVOLVED
try:
    sample = extract_text_from_pdf("/content/bug-bounty-bootcamp.pdf", "bug-bounty-bootcamp")
    print(f"✅ Success! Extracted {len(sample)} characters")
    print(f"First 200 chars: {sample[:200]}...")
except Exception as e:
    print(f"❌ ERROR: {str(e)}")

✅ Success! Extracted 843627 characters
First 200 chars: Li
Bug
Bounty
Bootcamp
Bug Bounty Bootcamp
The Guide to Finding and Reporting
Web Vulnerabilities
Vickie Li

BUG BOUNTY BOOTCAMP

B U G B O U N T Y
B O O T C A M P
The Guide to Finding and
Reporting W...


In [23]:
def process_pdf(pdf_path, book_title):
    """COMPLETELY FITZ-FREE PDF PROCESSING"""
    book_collection = f"rag_{book_title}"
    collections = client.list_collections()
    if book_collection in collections:
        print(f"Collection {book_collection} already exists")
        return

    # Create a seperate collection for new book
    if not create_collection(book_collection):
      return
    # 1. Extract text using pdfplumber (no fitz!)
    full_text = extract_text_from_pdf(pdf_path, book_title)

    # 2. Split into chunks
    chunks = text_splitter.split_text(full_text)

    # 3. Generate embeddings
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(chunks, convert_to_tensor=False)

    # 4. Prepare for Milvus
    data = [{
        "embedding": emb.tolist(),
        "text": chunk,
        "book_title": book_title,
        "chunk_order": i,
        "section_id": f"{book_title}_ch{i//10}"
    } for i, (chunk, emb) in enumerate(zip(chunks, embeddings))]
    client.insert(book_collection, data)
    return book_collection

In [19]:
client.list_collections()

['rag_vulnbot_milvus']

In [20]:
try:
    client.drop_collection(collection_name='rag_vulnbot_milvus')
    print("Collection dropped successfully")
except Exception as e:
    print(f"Failed to drop collection: {e}")

Collection dropped successfully


In [24]:
# Do not use - in path and name
books = [
    {"path": "/content/Penetration-Testing-intro.pdf", "title": "Penetration testing : a hands-on introduction to hacking"}
    # Add more books here
]

# Ingest a book
process_pdf(
    "/content/bug_bounty_bootcamp.pdf",
    "bug_county_bootcamp"
)

process_pdf(
    "/content/pentesting_intro.pdf",
    "pentesting_intro"
)

# Search with context
print(search_with_context("nikto web app scanning", context_window=2))

Collection rag_bug_county_bootcamp already exists
✅ Collection 'rag_pentesting_intro' created with metadata fields.
Connected to Milvus Online!
No results found


In [30]:
print(search_with_context("nikto web app scanning", context_window=2))

Connected to Milvus Online!
No results found in collection: rag_bug_county_bootcamp
versions, and misconfigurations. To run Nikto against our Linux target,
we tell it which host to scan with the -h flag, as shown in Listing 6-10.
root@kali:/# nikto -h 192.168.20.11
- Nikto v2.1.5
---------------------------------------------------------------------------
+ Target IP: 192.168.20.11
+ Target Hostname: 192.168.20.11
+ Target Port: 80
+ Start Time: 2015-12-28 21:31:38 (GMT-5)
---------------------------------------------------------------------------
+ Server: Apache/2.2.9 (Ubuntu) PHP/5.2.6-2ubuntu4.6 with Suhosin-Patch
--snip--
+ OSVDB-40478: /tikiwiki/tiki-graph_formula.php?w=1&h=1&s=1&min=1&max=2&f[]=x.
tan.phpinfo()&t=png&title=http://cirt.net/rfiinc.txt?: TikiWiki contains a
vulnerability which allows remote attackers to execute arbitrary PHP code. u
+ 6474 items checked: 2 error(s) and 7 item(s) reported on remote host
+ End Time: 2015-12-28 21:32:41 (GMT-5) (63 seconds)
Listing 6-1

Search for the question in the collection and retrieve the semantic top-3 matches.

Let's take a look at the search results of the query


### Use LLM to get a RAG response

Convert the retrieved documents into a string format.

Define system and user prompts for the Language Model. This prompt is assembled with the retrieved documents from Milvus.

In [ ]:
SYSTEM_PROMPT = """
Human: You are an AI assistant. You are able to find answers to the questions from the contextual passage snippets provided.
"""
USER_PROMPT = f"""
Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.
<context>
{context}
</context>
<question>
{question}
</question>
"""

Use OpenAI ChatGPT to generate a response based on the prompts.

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

Milvus stores data in persistent storage as incremental logs, including inserted data (vector data, scalar data, and collection-specific schema) and metadata. Inserted data is stored in various object storage backends like MinIO, AWS S3, Google Cloud Storage, Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud Object Storage. Metadata generated within Milvus is stored in etcd.


## Quick Deploy

To learn about how to start an online demo with this tutorial, please refer to [the example application](https://github.com/milvus-io/bootcamp/tree/master/tutorials/quickstart/apps/rag_search_with_milvus).